# **New York City Yellow Taxi Data**

## Objective
In this case study you will be learning exploratory data analysis (EDA) with the help of a dataset on yellow taxi rides in New York City. This will enable you to understand why EDA is an important step in the process of data science and machine learning.

## **Problem Statement**
As an analyst at an upcoming taxi operation in NYC, you are tasked to use the 2023 taxi trip data to uncover insights that could help optimise taxi operations. The goal is to analyse patterns in the data that can inform strategic decisions to improve service efficiency, maximise revenue, and enhance passenger experience.

## Tasks
You need to perform the following steps for successfully completing this assignment:
1. Data Loading
2. Data Cleaning
3. Exploratory Analysis: Bivariate and Multivariate
4. Creating Visualisations to Support the Analysis
5. Deriving Insights and Stating Conclusions

---

**NOTE:** The marks given along with headings and sub-headings are cumulative marks for those particular headings/sub-headings.<br>

The actual marks for each task are specified within the tasks themselves.

For example, marks given with heading *2* or sub-heading *2.1* are the cumulative marks, for your reference only. <br>

The marks you will receive for completing tasks are given with the tasks.

Suppose the marks for two tasks are: 3 marks for 2.1.1 and 2 marks for 3.2.2, or
* 2.1.1 [3 marks]
* 3.2.2 [2 marks]

then, you will earn 3 marks for completing task 2.1.1 and 2 marks for completing task 3.2.2.


---

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is from 2009 to 2024. However, for this assignment, we will only be using the data from 2023.

The data for each month is present in a different parquet file. You will get twelve files for each of the months in 2023.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

###  Data Description
You can find the data description here: [Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

**Taxi Zones**

Each of the trip records contains a field corresponding to the location of the pickup or drop-off of the trip, populated by numbers ranging from 1-263.

These numbers correspond to taxi zones, which may be downloaded as a table or map/shapefile and matched to the trip records using a join.

This is covered in more detail in later sections.

---

## **1** Data Preparation

<font color = red>[5 marks]</font> <br>

### Import Libraries

In [0]:
# Import warnings
import warnings
warnings.filterwarnings("ignore")

In [0]:
# Import the libraries you will be using for analysis
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
# Recommended versions
# numpy version: 1.26.4
# pandas version: 2.2.2
# matplotlib version: 3.10.0
# seaborn version: 0.13.2

# Check versions
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)

### **1.1** Load the dataset
<font color = red>[5 marks]</font> <br>

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [0]:
# Try loading one file
df = pd.read_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/trip_records/2023-1.parquet')
df.info()

How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

---

**1.1.1** <font color = red>[5 marks]</font> <br>
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [0]:
# Sample the data
# It is recommmended to not load all the files at once to avoid memory overload
df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df_sampled = df.groupby(['pickup_date','pickup_hour'],group_keys=False).apply(lambda x: x.sample(frac = 0.05, random_state=42))
len(df_sampled)

In [0]:
# from google.colab import drive
# drive.mount('/content/drive')

In [0]:
# Take a small percentage of entries from each hour of every date.
# Iterating through the monthly data:
#   read a month file -> day -> hour: append sampled data -> move to next hour -> move to next day after 24 hours -> move to next month file
# Create a single dataframe for the year combining all the monthly data

# Select the folder having data files
import os

# Select the folder having data files
#os.chdir('/content/Assignments/EDA/data_NYC_Taxi/trip_records')
os.chdir('/Volumes/nyc_taxi_analytics/raw/landing_zone/trip_records/') #Databricks volume

# Create a list of all the twelve files to read
file_list = os.listdir()

# initialise an empty dataframe
df = pd.DataFrame()


# iterate through the list of files and sample one by one:
for file_name in file_list:
    try:
        # file path for the current file
        file_path = os.path.join(os.getcwd(), file_name)

        # Reading the current file
        df_current = pd.read_parquet(file_path)


        # We will store the sampled data for the current date in this df by appending the sampled data from each hour to this
        # After completing iteration through each date, we will append this data to the final dataframe.
        sampled_data = pd.DataFrame()

        # Loop through dates and then loop through every hour of each date

            # Iterate through each hour of the selected date

                # Sample 5% of the hourly data randomly

                # add data of this hour to the dataframe
        df_current['pickup_date'] = df_current['tpep_pickup_datetime'].dt.date
        df_current['pickup_hour'] = df_current['tpep_pickup_datetime'].dt.hour
        sampled_data = df_current.groupby(['pickup_date','pickup_hour'],group_keys=False).apply(lambda x: x.sample(frac = 0.05, random_state=42))
        # Concatenate the sampled data of all the dates to a single dataframe
        df = pd.concat([df,sampled_data]) # we initialised this empty DF earlier

    except Exception as e:
        print(f"Error reading file {file_name}: {e}")

After combining the data files into one DataFrame, convert the new DataFrame to a CSV or parquet file and store it to use directly.

Ideally, you can try keeping the total entries to around 250,000 to 300,000.

In [0]:
len(df)

In [0]:
# Store the df in csv/parquet
df.to_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/sampled_records/sampled_data.parquet')

## **2** Data Cleaning
<font color = red>[30 marks]</font> <br>

Now we can load the new data directly.

In [0]:
# Load the new data file

newdf = pd.read_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/sampled_records/sampled_data.parquet')

In [0]:
newdf.head()

In [0]:
newdf.info()

#### **2.1** Fixing Columns
<font color = red>[10 marks]</font> <br>

Fix/drop any columns as you seem necessary in the below sections

**2.1.1** <font color = red>[2 marks]</font> <br>

Fix the index and drop unnecessary columns

In [0]:
# Fix the index and drop any columns that are not needed
newdf = newdf.reset_index(drop=True)
newdf = newdf.drop(columns=['extra','improvement_surcharge','store_and_fwd_flag','mta_tax'])
newdf.head(5)


**2.1.2** <font color = red>[3 marks]</font> <br>
There are two airport fee columns. This is possibly an error in naming columns. Let's see whether these can be combined into a single column.

In [0]:
# Combine the two airport fee columns

newdf['airport_fee'] = newdf['airport_fee'].combine_first(newdf['Airport_fee'])
newdf = newdf.drop(columns=['Airport_fee'])
newdf.head(5)

**2.1.3** <font color = red>[5 marks]</font> <br>
Fix columns with negative (monetary) values

In [0]:
# check where values of fare amount are negative
newdf[newdf['fare_amount'] < 0].head(30)

Did you notice something different in the `RatecodeID` column for above records?

In [0]:
# Analyse RatecodeID for the negative fare amounts

#No negative fare_amount columns

In [0]:
# Find which columns have negative values

newdf[(newdf['fare_amount'] < 0) | (newdf['tip_amount'] < 0) | (newdf['tolls_amount'] < 0) | (newdf['airport_fee'] < 0)  | (newdf['congestion_surcharge'] < 0)].head(5)

In [0]:
# fix these negative values

newdf = newdf[~((newdf['fare_amount'] < 0) | (newdf['tip_amount'] < 0) | (newdf['tolls_amount'] < 0) | (newdf['airport_fee'] < 0) | (newdf['congestion_surcharge'] < 0))]
len(newdf)

### **2.2** Handling Missing Values
<font color = red>[10 marks]</font> <br>

**2.2.1**  <font color = red>[2 marks]</font> <br>
Find the proportion of missing values in each column




In [0]:
# Find the proportion of missing values in each column
newdf.isna().mean().round(2) * 100


**2.2.2**  <font color = red>[3 marks]</font> <br>
Handling missing values in `passenger_count`

In [0]:
# Display the rows with null values
# Impute NaN values in 'passenger_count'
newdf['passenger_count'] = newdf['passenger_count'].fillna(round(newdf[newdf['passenger_count'] != 0]['passenger_count'].mean(),0))

Did you find zeroes in passenger_count? Handle these.

**2.2.3**  <font color = red>[2 marks]</font> <br>
Handle missing values in `RatecodeID`

In [0]:
# Fix missing values in 'RatecodeID'
# All of these are if payment_type = 0 (invalid), hence removing these
newdf = newdf[newdf['RatecodeID'].notna()]

**2.2.4**  <font color = red>[3 marks]</font> <br>
Impute NaN in `congestion_surcharge`

In [0]:
# handle null values in congestion_surcharge

newdf['congestion_surcharge'] = newdf['congestion_surcharge'].fillna(round(newdf['congestion_surcharge'].mean(),1))


Are there missing values in other columns? Did you find NaN values in some other set of columns? Handle those missing values below.

In [0]:
# Handle any remaining missing values

newdf.isna().mean().round(2) * 100

### **2.3** Handling Outliers
<font color = red>[10 marks]</font> <br>

Before we start fixing outliers, let's perform outlier analysis.

In [0]:
# Describe the data and check if there are any potential outliers present
# Check for potential out of place values in various columns

newdf.describe()

**2.3.1**  <font color = red>[10 marks]</font> <br>
Based on the above analysis, it seems that some of the outliers are present due to errors in registering the trips. Fix the outliers.

Some points you can look for:
- Entries where `trip_distance` is nearly 0 and `fare_amount` is more than 300
- Entries where `trip_distance` and `fare_amount` are 0 but the pickup and dropoff zones are different (both distance and fare should not be zero for different zones)
- Entries where `trip_distance` is more than 250  miles.
- Entries where `payment_type` is 0 (there is no payment_type 0 defined in the data dictionary)

These are just some suggestions. You can handle outliers in any way you wish, using the insights from above outlier analysis.

How will you fix each of these values? Which ones will you drop and which ones will you replace?

First, let us remove 7+ passenger counts as there are very less instances.

In [0]:
# remove passenger_count > 6
newdf = newdf[newdf['passenger_count'] <= 6]

In [0]:
# Continue with outlier handling
newdf = newdf[(newdf['RatecodeID'] >= 1) & (newdf['RatecodeID'] <= 6)]


In [0]:
LL = newdf['trip_distance'].quantile(0.01)
UL = newdf['trip_distance'].quantile(0.99)

newdf['trip_distance'] = newdf['trip_distance'].clip(lower=LL, upper=UL)

In [0]:
#This seem erroneous data of 36 rows
newdf = newdf[~((newdf['trip_distance'] <= 1) & (newdf['fare_amount'] >= 300))]

In [0]:
newdf = newdf[~(
    (newdf['trip_distance'] == 0) & 
    (newdf['fare_amount']   == 0) & 
    (newdf['PULocationID']  != newdf['DOLocationID'])
)]

In [0]:
newdf = newdf[newdf['total_amount'] >= 0].reset_index(drop=True)
newdf.head(2)

In [0]:
newdf.describe()

In [0]:
# Do any columns need standardising?
# cols_to_scale = ['trip_distance', 'fare_amount', 'tip_amount',
#                  'tolls_amount', 'total_amount', 'airport_fee']

# for col in cols_to_scale:
#     mean = newdf[col].mean()
#     std  = newdf[col].std()
#     newdf[col] = (newdf[col] - mean) / std

# newdf.head(5)


In [0]:
newdf.to_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/cleaned_copy/cleaned_nyc_taxi_data.parquet')

In [0]:
len(newdf)

## **3** Exploratory Data Analysis
<font color = red>[90 marks]</font> <br>

In [0]:
newdf = pd.read_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/cleaned_copy/cleaned_nyc_taxi_data.parquet')
newdf.columns.tolist()

In [0]:
newdf.info()

In [0]:
newdf.head()

#### **3.1** General EDA: Finding Patterns and Trends
<font color = red>[40 marks]</font> <br>

**3.1.1** <font color = red>[3 marks]</font> <br>
Categorise the varaibles into Numerical or Categorical.
* `VendorID`:
* `tpep_pickup_datetime`:
* `tpep_dropoff_datetime`:
* `passenger_count`:
* `trip_distance`:
* `RatecodeID`:
* `PULocationID`:
* `DOLocationID`:
* `payment_type`:
* `pickup_hour`:
* `trip_duration`:


The following monetary parameters belong in the same category, is it categorical or numerical?


* `fare_amount`
* `extra`
* `mta_tax`
* `tip_amount`
* `tolls_amount`
* `improvement_surcharge`
* `total_amount`
* `congestion_surcharge`
* `airport_fee`

##### Temporal Analysis

**3.1.2** <font color = red>[5 marks]</font> <br>
Analyse the distribution of taxi pickups by hours, days of the week, and months.

In [0]:
hourly_pickups = newdf.groupby('pickup_hour').size().reset_index(name='pickup_count')

sns.lineplot(data=hourly_pickups, x='pickup_hour', y='pickup_count', marker='o')
plt.title('Hourly Trends in Taxi Pickups')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Pickups')
plt.xticks(range(0, 24))
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
# Find and show the daily trends in taxi pickups (days of the week)

newdf['day_of_week'] = newdf['tpep_pickup_datetime'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_of_week_pickups = newdf.groupby('day_of_week').size().reset_index(name='pickup_count')
day_of_week_pickups['day_of_week'] = pd.Categorical(day_of_week_pickups['day_of_week'], categories=day_order, ordered=True)
day_of_week_pickups = day_of_week_pickups.sort_values('day_of_week')

sns.lineplot(data=day_of_week_pickups, x='day_of_week', y='pickup_count', marker='o')
plt.title('Taxi Pickup Trends by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Number of Pickups')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
# Show the monthly trends in pickups
newdf['month'] = newdf['tpep_pickup_datetime'].dt.month_name()

month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

monthly_pickups = newdf.groupby('month').size().reset_index(name='pickup_count')
monthly_pickups['month'] = pd.Categorical(monthly_pickups['month'], categories=month_order, ordered=True)
monthly_pickups = monthly_pickups.sort_values('month')

sns.lineplot(data=monthly_pickups, x='month', y='pickup_count', marker='o')
plt.title('Taxi Pickup Trends by Month')
plt.xlabel('Month')
plt.ylabel('Number of Pickups')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


##### Financial Analysis

Take a look at the financial parameters like `fare_amount`, `tip_amount`, `total_amount`, and also `trip_distance`. Do these contain zero/negative values?

In [0]:
# Analyse the above parameters
newdf[(newdf['trip_distance'] <= 0) & (newdf['PULocationID'] == newdf['DOLocationID'])]

In [0]:
newdf[newdf['fare_amount'] <=0 ]

In [0]:
newdf[newdf['tip_amount'] <=0 ]

Do you think it is beneficial to create a copy DataFrame leaving out the zero values from these?

**3.1.3** <font color = red>[2 marks]</font> <br>
Filter out the zero values from the above columns.

**Note:** The distance might be 0 in cases where pickup and drop is in the same zone. Do you think it is suitable to drop such cases of zero distance?

In [0]:
# Create a df with non zero entries for the selected parameters.
financial_df = newdf[((newdf['total_amount'] > 0) & (newdf['trip_distance'] > 0) & (newdf['fare_amount'] > 0))].copy()


**3.1.4** <font color = red>[3 marks]</font> <br>
Analyse the monthly revenue (`total_amount`) trend

In [0]:
financial_df.columns

In [0]:
# Group data by month and analyse monthly revenue
financial_df['month'] = financial_df['tpep_pickup_datetime'].dt.month_name()
month_order = ['January', 'February', 'March', 'April', 'May', 'June','July', 'August', 'September', 'October', 'November', 'December']

monthly_revenue = financial_df.groupby('month')['total_amount'].mean().reset_index()
monthly_revenue['month'] = pd.Categorical(monthly_revenue['month'], categories=month_order, ordered=True)
monthly_revenue = monthly_revenue.sort_values('month')

sns.lineplot(data=monthly_revenue, x='month', y='total_amount', marker='o')
plt.title('Monthly Trend in Total Amount')
plt.xlabel('Month')
plt.ylabel('Average Total Amount ($)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**3.1.5** <font color = red>[3 marks]</font> <br>
Show the proportion of each quarter of the year in the revenue

In [0]:
# Calculate proportion of each quarter

financial_df['quarter'] = financial_df['tpep_pickup_datetime'].dt.quarter
quarterly_revenue = financial_df.groupby('quarter')['total_amount'].sum().reset_index()
quarterly_revenue['revenue_proportion_%'] = ((quarterly_revenue['total_amount'] / quarterly_revenue['total_amount'].sum()) * 100).round(2)

display(quarterly_revenue)

**3.1.6** <font color = red>[3 marks]</font> <br>
Visualise the relationship between `trip_distance` and `fare_amount`. Also find the correlation value for these two.

**Hint:** You can leave out the trips with trip_distance = 0

In [0]:
# Show how trip fare is affected by distance
correlation = financial_df['trip_distance'].corr(financial_df['fare_amount'])
print(f"Correlation between trip_distance and fare_amount: {correlation:.4f}")


In [0]:
financial_df = financial_df[financial_df['trip_distance'] > 0]
sns.scatterplot(data=financial_df, x='trip_distance', y='fare_amount', alpha=0.3)
plt.title(f'Trip Distance vs Fare Amount')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Fare Amount ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
sns.regplot(data=financial_df, x='trip_distance', y='fare_amount',scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.title(f'Trip Distance vs Fare Amount')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Fare Amount ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

**3.1.7** <font color = red>[5 marks]</font> <br>
Find and visualise the correlation between:
1. `fare_amount` and trip duration (pickup time to dropoff time)
2. `fare_amount` and `passenger_count`
3. `tip_amount` and `trip_distance`

In [0]:
# Show relationship between fare and trip duration

financial_df['trip_duration'] = (financial_df['tpep_dropoff_datetime'] - financial_df['tpep_pickup_datetime']).dt.total_seconds() / 60
sns.scatterplot(data=financial_df[financial_df['trip_duration'] > 0], x='trip_duration', y='fare_amount', alpha=0.3)
plt.title(f'Trip Duration vs Fare Amount')
plt.xlabel('Trip Duration (minutes)')
plt.ylabel('Fare Amount ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [0]:
correlation = financial_df['trip_duration'].corr(financial_df['fare_amount'])
print(f"Correlation between trip_duration and fare_amount: {correlation:.4f}")

In [0]:
# Show relationship between fare and number of passengers
financial_df = financial_df[financial_df['passenger_count'] > 0]
sns.boxplot(data=financial_df, x='passenger_count', y='fare_amount')
plt.title(f'Number of passengers vs Fare Amount')
plt.xlabel('Number of passengers')
plt.ylabel('Fare Amount ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [0]:
correlation = financial_df['passenger_count'].corr(financial_df['fare_amount'])
print(f"Correlation between passenger_count and fare_amount: {correlation:.4f}")

In [0]:
# Show relationship between tip and trip distance

sns.regplot(data=financial_df, x='trip_distance', y='tip_amount',scatter_kws={'alpha': 0.3}, line_kws={'color': 'red'})
plt.title(f'Trip Distance vs Tip Amount')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Tip Amount ($)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [0]:
correlation = financial_df['trip_distance'].corr(financial_df['tip_amount'])
print(f"Correlation between trip_distance and tip_amount: {correlation:.4f}")

**3.1.8** <font color = red>[3 marks]</font> <br>
Analyse the distribution of different payment types (`payment_type`)

In [0]:
# Analyse the distribution of different payment types (payment_type).
payment_counts = financial_df['payment_type'].value_counts().reset_index()
payment_counts.columns = ['payment_type', 'count']
payment_counts['proportion_%'] = ((payment_counts['count'] / payment_counts['count'].sum()) * 100).round(4)

payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown', 6: 'Voided trip'}
payment_counts['payment_label'] = payment_counts['payment_type'].map(payment_labels)


In [0]:
payment_counts.head()

- 1= Credit card
- 2= Cash
- 3= No charge
- 4= Dispute



In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=payment_counts, x='payment_label', y='count', ax=axes[0])
axes[0].set_title('Payment Type Distribution')
axes[0].set_xlabel('Payment Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].pie(
    payment_counts['count'],
    autopct   = '%1.1f%%',
    startangle= 90,
    pctdistance= 1.10,
    explode   = [0.10] * len(payment_counts)
)

axes[1].legend(
    payment_counts['payment_label'],
    loc            = 'upper right',
    bbox_to_anchor = (1.3, 1.0),
    fontsize       = 9
)
axes[1].set_title('Payment Type Proportion')

plt.tight_layout()
plt.show()

##### Geographical Analysis

For this, you have to use the *taxi_zones.shp* file from the *taxi_zones* folder.

There would be multiple files inside the folder (such as *.shx, .sbx, .sbn* etc). You do not need to import/read any of the files other than the shapefile, *taxi_zones.shp*.

Do not change any folder structure - all the files need to be present inside the folder for it to work.

The folder structure should look like this:
```
Taxi Zones
|- taxi_zones.shp.xml
|- taxi_zones.prj
|- taxi_zones.sbn
|- taxi_zones.shp
|- taxi_zones.dbf
|- taxi_zones.shx
|- taxi_zones.sbx

 ```

 You only need to read the `taxi_zones.shp` file. The *shp* file will utilise the other files by itself.

We will use the *GeoPandas* library for geopgraphical analysis
```
import geopandas as gpd
```

More about geopandas and shapefiles: [About](https://geopandas.org/en/stable/about.html)


Reading the shapefile is very similar to *Pandas*. Use `gpd.read_file()` function to load the data (*taxi_zones.shp*) as a GeoDataFrame. Documentation: [Reading and Writing Files](https://geopandas.org/en/stable/docs/user_guide/io.html)

In [0]:
# !pip install geopandas

**3.1.9** <font color = red>[2 marks]</font> <br>
Load the shapefile and display it.

In [0]:
import geopandas as gpd


# Read the shapefile using geopandas
zones = gpd.read_file('/Volumes/nyc_taxi_analytics/raw/landing_zone/taxi_zones/taxi_zones.shp')# read the .shp file using gpd
display(zones.head())

Now, if you look at the DataFrame created, you will see columns like: `OBJECTID`,`Shape_Leng`, `Shape_Area`, `zone`, `LocationID`, `borough`, `geometry`.
<br><br>

Now, the `locationID` here is also what we are using to mark pickup and drop zones in the trip records.

The geometric parameters like shape length, shape area and geometry are used to plot the zones on a map.

This can be easily done using the `plot()` method.

In [0]:
zones.info()
fig, ax = plt.subplots(figsize=(10, 8))
zones.plot(ax=ax)
plt.title('NYC Taxi Zones')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()  

Now, you have to merge the trip records and zones data using the location IDs.



**3.1.10** <font color = red>[3 marks]</font> <br>
Merge the zones data into trip data using the `locationID` and `PULocationID` columns.

In [0]:
# Merge zones and trip records using locationID and PULocationID
merged_df = financial_df.merge(zones, left_on='PULocationID', right_on='LocationID', how='left')
merged_df

**3.1.11** <font color = red>[3 marks]</font> <br>
Group data by location IDs to find the total number of trips per location ID

In [0]:
# Group data by location and calculate the number of trips
location_trips = merged_df['LocationID'].value_counts().reset_index()
location_trips.columns = ['LocationID', 'trip_count']
location_trips.head(10)


**3.1.12** <font color = red>[2 marks]</font> <br>
Now, use the grouped data to add number of trips to the GeoDataFrame.

We will use this to plot a map of zones showing total trips per zone.

In [0]:
# Merge trip counts back to the zones GeoDataFrame

zones_with_trips = zones.merge(location_trips, on='LocationID', how='left')
zones_with_trips.head(10)

The next step is creating a color map (choropleth map) showing zones by the number of trips taken.

Again, you can use the `zones.plot()` method for this. [Plot Method GPD](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html#geopandas.GeoDataFrame.plot)

But first, you need to define the figure and axis for the plot.

`fig, ax = plt.subplots(1, 1, figsize = (12, 10))`

This function creates a figure (fig) and a single subplot (ax)

---

After setting up the figure and axis, we can proceed to plot the GeoDataFrame on this axis. This is done in the next step where we use the plot method of the GeoDataFrame.

You can define the following parameters in the `zones.plot()` method:
```
column = '',
ax = ax,
legend = True,
legend_kwds = {'label': "label", 'orientation': "<horizontal/vertical>"}
```

To display the plot, use `plt.show()`.

**3.1.13** <font color = red>[3 marks]</font> <br>
Plot a color-coded map showing zone-wise trips

In [0]:
len(zones_with_trips[zones_with_trips['trip_count'].isna()])

In [0]:
zones_with_trips = zones_with_trips[zones_with_trips['trip_count'].notna()]
zones_with_trips.describe()

In [0]:
# Define figure and axis
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

zones_with_trips.plot(
    column      = 'trip_count',
    ax          = ax,
    legend      = True,
    legend_kwds = {'label': 'Number of Trips', 'orientation': 'horizontal'}
)

# Plot the map and display it
plt.title('NYC Taxi Pickup Density by Zone')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()


In [0]:
# can you try displaying the zones DF sorted by the number of trips?

zones_with_trips.sort_values('trip_count',ascending=False).head(30)

In [0]:
merged_df.columns

Here we have completed the temporal, financial and geographical analysis on the trip records.

**Compile your findings from general analysis below:**

You can consider the following points:

* Busiest hours, days and months
* Trends in revenue collected
* Trends in quarterly revenue
* How fare depends on trip distance, trip duration and passenger counts
* How tip amount depends on trip distance
* Busiest zones


#### **3.2** Detailed EDA: Insights and Strategies
<font color = red>[50 marks]</font> <br>

Having performed basic analyses for finding trends and patterns, we will now move on to some detailed analysis focussed on operational efficiency, pricing strategies, and customer experience.

##### Operational Efficiency

Analyze variations by time of day and location to identify bottlenecks or inefficiencies in routes

**3.2.1** <font color = red>[3 marks]</font> <br>
Identify slow routes by calculating the average time taken by cabs to get from one zone to another at different hours of the day.

Speed on a route *X* for hour *Y* = (*distance of the route X / average trip duration for hour Y*)

In [0]:
# Find routes which have the slowest speeds at different times of the day

route_speed = merged_df.groupby(['PULocationID', 'DOLocationID', 'pickup_hour']).agg(
    avg_distance = ('trip_distance', 'mean'),
    avg_duration = ('trip_duration', 'mean'),
    trip_count   = ('trip_distance', 'count')
).reset_index()
route_speed.head()

In [0]:
route_speed['speed'] = route_speed['avg_distance'] / route_speed['avg_duration']
route_speed['speed_mph'] = route_speed['speed'] * 60
route_speed = route_speed[(route_speed['avg_duration'] > 0) & (route_speed['avg_distance'] > 0)]
slow_routes = route_speed.sort_values('speed_mph', ascending=True)
slow_routes.head(10)

In [0]:
slow_routes = slow_routes.merge(zones[['LocationID', 'zone']], left_on='PULocationID', right_on='LocationID', how='left').rename(columns={'zone': 'pickup_zone'}).drop(columns='LocationID')

slow_routes = slow_routes.merge(zones[['LocationID', 'zone']], left_on='DOLocationID', right_on='LocationID', how='left').rename(columns={'zone': 'dropoff_zone'}).drop(columns='LocationID')

slow_routes[['pickup_zone', 'dropoff_zone', 'pickup_hour', 'avg_distance', 'avg_duration', 'speed_mph', 'trip_count']].sort_values('speed_mph').head(10)

In [0]:
slow_zone_counts = slow_routes.groupby('pickup_zone').size().reset_index(name='slow_route_count')
zones_slow = zones.merge(slow_zone_counts, left_on='zone', right_on='pickup_zone', how='left')
zones_slow['slow_route_count'] = zones_slow['slow_route_count'].fillna(0)

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

zones_slow.plot(
    column      = 'slow_route_count',
    ax          = ax,
    legend      = True,
    cmap        = 'Reds',
    edgecolor   = 'black',
    linewidth   = 0.3,
    legend_kwds = {'label': 'Number of Slow Routes', 'orientation': 'horizontal'}
)

plt.title('Slow Route Density by Pickup Zone\n(Routes with Avg Speed < 10 mph)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

In [0]:
zones_slow.head()

How does identifying high-traffic, high-demand routes help us?

**3.2.2** <font color = red>[3 marks]</font> <br>
Calculate the number of trips at each hour of the day and visualise them. Find the busiest hour and show the number of trips for that hour.

In [0]:
# Visualise the number of trips per hour and find the busiest hour
hourly_trips = merged_df.groupby('pickup_hour').size().reset_index(name='trip_count')


In [0]:
sampling_ratio = 0.05
hourly_trips['actual_trip_count'] = (hourly_trips['trip_count'] / sampling_ratio).astype(int)
busiest_hour = hourly_trips.loc[hourly_trips['trip_count'].idxmax()]
print(f"Busiest Hour        : {int(busiest_hour['pickup_hour'])}:00")
print(f"Sampled Trips       : {int(busiest_hour['trip_count'])}")
print(f"Actual Trips (scaled): {int(busiest_hour['actual_trip_count'])}")

In [0]:
sns.lineplot(data=hourly_trips, x='pickup_hour', y='trip_count', marker='o')
plt.axvline(x=busiest_hour['pickup_hour'], color='red', linestyle='--', label=f"Busiest Hour: {int(busiest_hour['pickup_hour'])}:00")
plt.title('Number of Trips by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips (Sampled)')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

Remember, we took a fraction of trips. To find the actual number, you have to scale the number up by the sampling ratio.

**3.2.3** <font color = red>[2 mark]</font> <br>
Find the actual number of trips in the five busiest hours

In [0]:
# Scale up the number of trips

# Fill in the value of your sampling fraction and use that to scale up the numbers
sample_fraction = 0.05

top5_busy_hours = hourly_trips.nlargest(5, 'trip_count').copy()
top5_busy_hours['actual_trip_count'] = (top5_busy_hours['trip_count'] / sample_fraction).astype(int)
display(top5_busy_hours[['pickup_hour', 'trip_count', 'actual_trip_count']])

**3.2.4** <font color = red>[3 marks]</font> <br>
Compare hourly traffic pattern on weekdays. Also compare for weekend.

In [0]:
# Compare traffic trends for the week days and weekends

merged_df['day_of_week'] = merged_df['tpep_pickup_datetime'].dt.dayofweek
merged_df['day_type']    = merged_df['day_of_week'].apply(lambda x: 'Weekend' if x >= 5 else 'Weekday')

In [0]:
hourly_day_type = merged_df.groupby(['pickup_hour', 'day_type']).size().reset_index(name='trip_count')

In [0]:
hourly_day_type['actual_trip_count'] = (hourly_day_type['trip_count'] / sample_fraction).astype(int)

In [0]:
sns.lineplot(data=hourly_day_type, x='pickup_hour', y='actual_trip_count', hue='day_type', marker='o')
plt.title('Hourly Traffic Pattern — Weekday vs Weekend')
plt.xlabel('Hour of Day')
plt.ylabel('Actual Number of Trips')
plt.xticks(range(0, 24))
plt.legend(title='Day Type')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

What can you infer from the above patterns? How will finding busy and quiet hours for each day help us?

**3.2.5** <font color = red>[3 marks]</font> <br>
Identify top 10 zones with high hourly pickups. Do the same for hourly dropoffs. Show pickup and dropoff trends in these zones.

In [0]:
# Find top 10 pickup and dropoff zones

top10_pickup_zones = merged_df.groupby('zone').size().reset_index(name='pickup_count').nlargest(10, 'pickup_count')
top10_pickup_zones

In [0]:
dropoff_df = merged_df.merge(zones[['LocationID', 'zone']], left_on='DOLocationID', right_on='LocationID', how='left', suffixes=('', '_dropoff')).rename(columns={'zone_dropoff': 'dropoff_zone'})

top10_dropoff_zones = dropoff_df.groupby('dropoff_zone').size().reset_index(name='dropoff_count').nlargest(10, 'dropoff_count')

top10_dropoff_zones

In [0]:
top10_pickup_names  = top10_pickup_zones['zone'].tolist()
top10_dropoff_names = top10_dropoff_zones['dropoff_zone'].tolist()

hourly_pickup_trends = merged_df[merged_df['zone'].isin(top10_pickup_names)].groupby(['pickup_hour', 'zone']).size().reset_index(name='trip_count')

hourly_dropoff_trends = dropoff_df[dropoff_df['dropoff_zone'].isin(top10_dropoff_names)].groupby(['pickup_hour', 'dropoff_zone']).size().reset_index(name='trip_count')


In [0]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

sns.lineplot(data=hourly_pickup_trends, x='pickup_hour', y='trip_count',hue='zone', marker='o', ax=axes[0])
axes[0].set_title('Hourly Pickup Trends — Top 10 Zones')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Pickups')
axes[0].set_xticks(range(0, 24))
axes[0].legend(title='Zone', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.lineplot(data=hourly_dropoff_trends, x='pickup_hour', y='trip_count',hue='dropoff_zone', marker='o', ax=axes[1])
axes[1].set_title('Hourly Dropoff Trends — Top 10 Zones')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Number of Dropoffs')
axes[1].set_xticks(range(0, 24))
axes[1].legend(title='Zone', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

**3.2.6** <font color = red>[3 marks]</font> <br>
Find the ratio of pickups and dropoffs in each zone. Display the 10 highest (pickup/drop) and 10 lowest (pickup/drop) ratios.

In [0]:
# Find the top 10 and bottom 10 pickup/dropoff ratios

pickup_counts = merged_df.groupby('zone').size().reset_index(name='pickup_count')
dropoff_counts = dropoff_df.groupby('dropoff_zone').size().reset_index(name='dropoff_count').rename(columns={'dropoff_zone': 'zone'})

In [0]:
zone_ratio = pickup_counts.merge(dropoff_counts, on='zone', how='outer').fillna(0)

In [0]:
zone_ratio['pickup_dropoff_ratio'] = (zone_ratio['pickup_count'] / zone_ratio['dropoff_count']).round(4)

In [0]:
top10_highest = zone_ratio.nlargest(10, 'pickup_dropoff_ratio')
top10_highest[['zone', 'pickup_count', 'dropoff_count', 'pickup_dropoff_ratio']]

In [0]:
top10_lowest = zone_ratio.nsmallest(10, 'pickup_dropoff_ratio')
top10_lowest[['zone', 'pickup_count', 'dropoff_count', 'pickup_dropoff_ratio']]

**3.2.7** <font color = red>[3 marks]</font> <br>
Identify zones with high pickup and dropoff traffic during night hours (11PM to 5AM)

In [0]:
# During night hours (11pm to 5am) find the top 10 pickup and dropoff zones
# Note that the top zones should be of night hours and not the overall top zones

night_df = merged_df[merged_df['pickup_hour'].isin([23, 0, 1, 2, 3, 4, 5])]
top10_night_pickups = night_df.groupby('zone').size().reset_index(name='pickup_count').nlargest(10, 'pickup_count')
top10_night_pickups


In [0]:
night_dropoff_df = night_df.merge(zones[['LocationID', 'zone']],left_on='DOLocationID', right_on='LocationID',how='left', suffixes=('', '_dropoff')).rename(columns={'zone_dropoff': 'dropoff_zone'})
top10_night_dropoffs = night_dropoff_df.groupby('dropoff_zone').size().reset_index(name='dropoff_count').nlargest(10, 'dropoff_count')
top10_night_dropoffs

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Night pickup zones
sns.barplot(data=top10_night_pickups, x='pickup_count', y='zone', ax=axes[0], color='midnightblue')
axes[0].set_title('Top 10 Night Pickup Zones\n(11pm – 5am)')
axes[0].set_xlabel('Number of Pickups')
axes[0].set_ylabel('Zone')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Night dropoff zones
sns.barplot(data=top10_night_dropoffs, x='dropoff_count', y='dropoff_zone', ax=axes[1], color='darkred')
axes[1].set_title('Top 10 Night Dropoff Zones\n(11pm – 5am)')
axes[1].set_xlabel('Number of Dropoffs')
axes[1].set_ylabel('Zone')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

Now, let us find the revenue share for the night time hours and the day time hours. After this, we will move to deciding a pricing strategy.

**3.2.8** <font color = red>[2 marks]</font> <br>
Find the revenue share for nighttime and daytime hours.

In [0]:
# Filter for night hours (11 PM to 5 AM)
merged_df['time_of_day'] = merged_df['pickup_hour'].apply(lambda x: 'Nighttime' if x in [23, 0, 1, 2, 3, 4, 5] else 'Daytime')


In [0]:
revenue_share = merged_df.groupby('time_of_day')['total_amount'].agg(
    total_revenue = 'sum',
    avg_revenue   = 'mean',
    trip_count    = 'count'
).reset_index()

revenue_share['revenue_share_%'] = ((revenue_share['total_revenue'] / revenue_share['total_revenue'].sum()) * 100).round(2)

revenue_share

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(revenue_share['total_revenue'],
            labels    = revenue_share['time_of_day'],
            autopct   = '%1.1f%%',
            startangle= 90,
            colors    = ['steelblue', 'midnightblue'],
            explode   = [0.05, 0.05])
axes[0].set_title('Revenue Share\nDaytime vs Nighttime')

sns.barplot(data=revenue_share, x='time_of_day', y='avg_revenue',
            palette=['steelblue', 'midnightblue'], ax=axes[1])
axes[1].set_title('Average Revenue per Trip\nDaytime vs Nighttime')
axes[1].set_xlabel('Time of Day')
axes[1].set_ylabel('Average Revenue ($)')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
merged_df.columns

In [0]:
merged_df.drop(columns=['geometry']).to_parquet('/Volumes/nyc_taxi_analytics/gold/analytics/gold_copy/gold_nyc_taxi_data.parquet')

In [0]:
merged_df = pd.read_parquet('/Volumes/nyc_taxi_analytics/gold/analytics/gold_copy/gold_nyc_taxi_data.parquet')

In [0]:
len(merged_df)

##### Pricing Strategy

**3.2.9** <font color = red>[2 marks]</font> <br>
For the different passenger counts, find the average fare per mile per passenger.

For instance, suppose the average fare per mile for trips with 3 passengers is 3 USD/mile, then the fare per mile per passenger will be 1 USD/mile.

In [0]:
# Analyse the fare per mile per passenger for different passenger counts

fare_per_mile = merged_df[merged_df['trip_distance'] > 0].groupby('passenger_count').agg(
    avg_fare     = ('fare_amount', 'mean'),
    avg_distance = ('trip_distance', 'mean'),
    trip_count   = ('trip_distance', 'count')
).reset_index()


In [0]:
fare_per_mile['fare_per_mile'] = (fare_per_mile['avg_fare'] / fare_per_mile['avg_distance']).round(4)

In [0]:
fare_per_mile['fare_per_mile_per_passenger'] = (fare_per_mile['fare_per_mile'] / fare_per_mile['passenger_count']).round(4)

fare_per_mile[['passenger_count', 'avg_fare', 'avg_distance', 'fare_per_mile', 'fare_per_mile_per_passenger']]

**3.2.10** <font color = red>[3 marks]</font> <br>
Find the average fare per mile by hours of the day and by days of the week

In [0]:
# Compare the average fare per mile for different days and for different times of the day

valid_df = merged_df[merged_df['trip_distance'] > 0].copy()
valid_df['fare_per_mile'] = valid_df['fare_amount'] / valid_df['trip_distance']

In [0]:
hourly_fare = valid_df.groupby('pickup_hour')['fare_per_mile'].mean().round(2).reset_index()
hourly_fare.columns = ['pickup_hour', 'avg_fare_per_mile']

In [0]:
hourly_fare.head(30)

In [0]:
valid_df['day_of_week'] = valid_df['tpep_pickup_datetime'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

daily_fare = valid_df.groupby('day_of_week')['fare_per_mile'].mean().round(2).reset_index()
daily_fare.columns = ['day_of_week', 'avg_fare_per_mile']
daily_fare['day_of_week'] = pd.Categorical(daily_fare['day_of_week'], categories=day_order, ordered=True)
daily_fare = daily_fare.sort_values('day_of_week')
daily_fare.head(10)

**3.2.11** <font color = red>[3 marks]</font> <br>
Analyse the average fare per mile for the different vendors for different hours of the day

In [0]:
# Compare fare per mile for different vendors

valid_df['vendor_name'] = valid_df['VendorID'].map({1: 'Creative Mobile Technologies', 2: 'VeriFone Inc.'})
vendor_hourly_fare = valid_df.groupby(['vendor_name', 'pickup_hour'])['fare_per_mile'].mean().round(2).reset_index()
vendor_hourly_fare.columns = ['vendor_name', 'pickup_hour', 'avg_fare_per_mile']
vendor_hourly_fare.head(5)

In [0]:
sns.barplot(data=vendor_hourly_fare, x='pickup_hour', y='avg_fare_per_mile',hue='vendor_name')
plt.title('Avg Fare per Mile by Vendor')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Fare per Mile ($/mile)')
plt.legend(title='Vendor')
plt.legend(loc='upper right')    
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

**3.2.12** <font color = red>[5 marks]</font> <br>
Compare the fare rates of the different vendors in a tiered fashion. Analyse the average fare per mile for distances upto 2 miles. Analyse the fare per mile for distances from 2 to 5 miles. And then for distances more than 5 miles.


In [0]:
# Defining distance tiers

conditions = [
    valid_df['trip_distance'] <= 2,
    valid_df['trip_distance'].between(2, 5),
    valid_df['trip_distance'] > 5
]
choices = ['Short (≤ 2 miles)', 'Medium (2–5 miles)', 'Long (> 5 miles)']
valid_df['distance_tier'] = np.select(conditions, choices)
valid_df.head()

In [0]:
tiered_fare = valid_df.groupby(['distance_tier', 'vendor_name'])['fare_per_mile'].mean().round(2).reset_index()
tiered_fare.head()

In [0]:
tier_order = ['Short (≤ 2 miles)', 'Medium (2–5 miles)', 'Long (> 5 miles)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, tier in zip(axes, tier_order):
    data = tiered_fare[tiered_fare['distance_tier'] == tier]
    sns.barplot(data=data, x='vendor_name', y='fare_per_mile', ax=ax)
    ax.set_title(f'Avg Fare per Mile\n{tier}')
    ax.set_xlabel('Vendor')
    ax.set_ylabel('Avg Fare per Mile ($/mile)')
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

##### Customer Experience and Other Factors

**3.2.13** <font color = red>[5 marks]</font> <br>
Analyse average tip percentages based on trip distances, passenger counts and time of pickup. What factors lead to low tip percentages?

In [0]:
#  Analyze tip percentages based on distances, passenger counts and pickup times

valid_df['tip_percentage'] = (valid_df['tip_amount'] / valid_df['fare_amount'] * 100).round(2)

In [0]:
conditions = [
    valid_df['trip_distance'] <= 2,
    valid_df['trip_distance'].between(2, 5),
    valid_df['trip_distance'] > 5
]
choices            = ['Short (≤ 2 miles)', 'Medium (2–5 miles)', 'Long (> 5 miles)']
tier_order         = ['Short (≤ 2 miles)', 'Medium (2–5 miles)', 'Long (> 5 miles)']
valid_df['distance_tier'] = np.select(conditions, choices)
valid_df.head()

In [0]:
tip_by_distance = valid_df.groupby('distance_tier')['tip_percentage'].mean().round(2).reset_index()
tip_by_distance['distance_tier'] = pd.Categorical(tip_by_distance['distance_tier'],categories=tier_order, ordered=True)
tip_by_distance = tip_by_distance.sort_values('distance_tier')
tip_by_distance.head()

In [0]:
tip_by_passengers = valid_df.groupby('passenger_count')['tip_percentage'].mean().round(2).reset_index()
tip_by_passengers.head()

In [0]:
tip_by_hour = valid_df.groupby('pickup_hour')['tip_percentage'].mean().round(2).reset_index()
tip_by_hour.head(30)

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Plot 1 — By distance tier
sns.barplot(data=tip_by_distance, x='distance_tier', y='tip_percentage', ax=axes[0])
axes[0].set_title('Avg Tip %\nby Distance Tier')
axes[0].set_xlabel('Distance Tier')
axes[0].set_ylabel('Avg Tip (%)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Plot 2 — By passenger count
sns.barplot(data=tip_by_passengers, x='passenger_count', y='tip_percentage', ax=axes[1])
axes[1].set_title('Avg Tip %\nby Passenger Count')
axes[1].set_xlabel('Passenger Count')
axes[1].set_ylabel('Avg Tip (%)')
axes[1].grid(True, linestyle='--', alpha=0.5)

# Plot 3 — By hour of day
sns.lineplot(data=tip_by_hour, x='pickup_hour', y='tip_percentage', marker='o', ax=axes[2])
axes[2].set_title('Avg Tip %\nby Hour of Day')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Avg Tip (%)')
axes[2].set_xticks(range(0, 24))
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [0]:
cash_trips = valid_df[valid_df['payment_type'] == 2]['tip_percentage'].mean()
print(cash_trips)

In [0]:
zero_tip = valid_df[valid_df['tip_amount'] == 0]

zero_tip_profile = pd.DataFrame({
    'avg_distance'     : [zero_tip['trip_distance'].mean()],
    'avg_fare'         : [zero_tip['fare_amount'].mean()],
    'avg_passengers'   : [zero_tip['passenger_count'].mean()],
    'cash_payment_%'   : [(zero_tip['payment_type'] == 2).mean() * 100],
    'night_trip_%'     : [(zero_tip['time_of_day'] == 'Nighttime').mean() * 100]
}).round(2)

zero_tip_profile.head()

Additional analysis [optional]: Let's try comparing cases of low tips with cases of high tips to find out if we find a clear aspect that drives up the tipping behaviours

In [0]:
# Compare trips with tip percentage < 10% to trips with tip percentage > 25%



**3.2.14** <font color = red>[3 marks]</font> <br>
Analyse the variation of passenger count across hours and days of the week.

In [0]:
# See how passenger count varies across hours and days
passengers_by_hour = merged_df.groupby('pickup_hour')['passenger_count'].mean().round(2).reset_index()



In [0]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
merged_df['day_of_week'] = merged_df['tpep_pickup_datetime'].dt.day_name()
passengers_by_day = merged_df.groupby('day_of_week')['passenger_count'].mean().round(2).reset_index()
passengers_by_day['day_of_week'] = pd.Categorical(passengers_by_day['day_of_week'],categories=day_order, ordered=True)
passengers_by_day = passengers_by_day.sort_values('day_of_week')
passengers_by_day.head()

In [0]:
heatmap_data = merged_df.groupby(['day_of_week', 'pickup_hour'])['passenger_count'].mean().round(2).unstack()
heatmap_data.index = pd.CategoricalIndex(heatmap_data.index,categories=day_order, ordered=True)
heatmap_data = heatmap_data.sort_index()
heatmap_data.head(10)

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Plot 1 — By hour
sns.lineplot(data=passengers_by_hour, x='pickup_hour', y='passenger_count',marker='o', ax=axes[0])
axes[0].set_title('Avg Passenger Count\nby Hour of Day')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Passenger Count')
axes[0].set_xticks(range(0, 24))
axes[0].grid(True, linestyle='--', alpha=0.5)

# Plot 2 — By day of week
sns.barplot(data=passengers_by_day, x='day_of_week', y='passenger_count', ax=axes[1])
axes[1].set_title('Avg Passenger Count\nby Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Passenger Count')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Plot 3 — Heatmap (Hour x Day)
fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Passenger Count'})
ax.set_title('Avg Passenger Count — Hour of Day vs Day of Week')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.show()

**3.2.15** <font color = red>[2 marks]</font> <br>
Analyse the variation of passenger counts across zones

In [0]:
# How does passenger count vary across zones
passengers_by_zone = merged_df.groupby('zone')['passenger_count'].mean().round(2).reset_index()
passengers_by_zone.columns = ['zone', 'avg_passenger_count']
passengers_by_zone.head()


In [0]:
# For a more detailed analysis, we can use the zones_with_trips GeoDataFrame
# Create a new column for the average passenger count in each zone.

zones_with_trips = zones_with_trips.merge(passengers_by_zone, on='zone', how='left')
zones_with_trips['avg_passenger_count'] = zones_with_trips['avg_passenger_count'].fillna(0)
zones_with_trips.sort_values('avg_passenger_count',ascending=False).head(10)

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

zones_with_trips.plot(
    column      = 'avg_passenger_count',
    ax          = ax,
    legend      = True,
    cmap        = 'YlOrRd',
    legend_kwds = {'label': 'Avg Passenger Count', 'orientation': 'horizontal'}
)

plt.title('Avg Passenger Count by Zone')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

Find out how often surcharges/extra charges are applied to understand their prevalance

**3.2.16** <font color = red>[5 marks]</font> <br>
Analyse the pickup/dropoff zones or times when extra charges are applied more frequently

In [0]:
# How often is each surcharge applied?



## **4** Conclusion
<font color = red>[15 marks]</font> <br>

### **4.1** Final Insights and Recommendations
<font color = red>[15 marks]</font> <br>

Conclude your analyses here. Include all the outcomes you found based on the analysis.

Based on the insights, frame a concluding story explaining suitable parameters such as location, time of the day, day of the week etc. to be kept in mind while devising a strategy to meet customer demand and optimise supply.

**4.1.1** <font color = red>[5 marks]</font> <br>
Recommendations to optimize routing and dispatching based on demand patterns and operational inefficiencies

**Demand Patterns**

Weekday mornings between 7–9am and evenings between 4–7pm represent the two primary peak windows. Wednesday and Thursday are the busiest days of the week with 3-6pm being the top 5 busiet hours of the day. With weekend nights between 10pm and 2am generating the highest average passenger counts per trip, reflecting group leisure and nightlife travel. Monthly analysis reveals Q3 as the strongest revenue quarter while Q1 shows a dip, indicating seasonal demand during rainy season. Notably, nighttime hours generate lower trip volumes but consistently higher average fares per trip, suggesting that night riders take longer cross-borough journeys and are less price-sensitive than daytime commuters.

**Operational Inefficiencies**

During peak hours of 8–9am and 5–7pm, slow routes across Midtown substantially reduce the number of trips a driver can complete per hour, compressing hourly earnings despite high demand. The minimum fare effect disproportionately inflates the cost per mile for short trips under 2 miles, creating a poor value perception among passengers and contributing to lower tip rates on these journeys. Additionally, cash payments accounting for approximately 18% of all trips render tip data invisible, introducing a systematic bias into any revenue or satisfaction analysis. Finally, the divergence in fare-per-mile between vendors Creative Mobile and VeriFone widens during peak hours, suggesting inconsistent surge pricing algorithms that warrant further auditing to ensure fare equity across the fleet.

**4.1.2** <font color = red>[5 marks]</font> <br>

Suggestions on strategically positioning cabs across different zones to make best use of insights uncovered by analysing trip trends across time, days and months.

**Strategic Cab Positioning Recommendations**

Cab positioning strategy should consider the weekday peak office hours and the weekend leisure. On weekdays, drivers should be pre-positioned in Penn station, Upper East Side, and key business corridors by 7am to capture the morning commuter surge between 7–9am. As demand subsides through mid-morning, drivers should migrate toward airport corridors serving JFK and LaGuardia, which maintain consistent demand throughout the day and offer high absolute fare values. A second repositioning toward Midtown and Upper East Side should be executed by 4pm ahead of the evening rush between 4–7pm, which represents the single highest demand window of the week.

On weekends, drivers should avoid over-concentrating in business districts, which experience significantly lower weekend demand, and instead position across hotel corridors, tourist zones, and entertainment districts through the afternoon. As Friday and Saturday nights progress beyond 10pm, proactive clustering in nightlife districts such as the Meatpacking District and Lower East Side becomes critical, as these zones generate the highest passenger counts per trip and the highest average fares of the entire week.

During Q3, maximum fleet availability should be maintained across all high-demand zones due to rainy season where usually demand surges, while Q1 can have a reduction in active supply during off-peak windows. Night hours across all seasons present a consistent opportunity — though trip volumes are lower, average fares are higher, making dedicated night-shift positioning in cross-borough and airport zones a strong revenue-per-hour strategy for drivers willing to operate beyond midnight.

**4.1.3** <font color = red>[5 marks]</font> <br>
Propose data-driven adjustments to the pricing strategy to maximize revenue while maintaining competitive rates with other vendors.


**Tiered Distance Pricing**

Analysis of fare-per-mile across distance tiers reveals that the short trips under 2 miles have disproportionately high per-mile costs due to the minimum fare structure, while long trips beyond 5 miles offer the lowest per-mile rates. This creates a poor value among short-trip passengers while not monetizing the long trips. A tiered distance pricing model should be introduced where the minimum fare threshold is marginally reduced to improve short-trip affordability and passenger retention, while a modest premium is applied to long trips beyond 5 miles that currently generate the highest absolute fares but lowest per-mile recovery. This rebalancing would improve fare equity across trip lengths without compromising overall revenue.

**Dynamic Surge Pricing Calibration**

Hourly demand analysis clearly identifies 7–9am and 4–7pm on weekdays and 10pm–2am on Friday and Saturday nights as peak demand windows with consistently high trip volumes. Surge pricing should be considered specifically for these windows, with the highest reserved for Friday and Saturday nights in nightlife zones where passenger price sensitivity is lower. Conversely, surge pricing should be suppressed during mid-morning and early afternoon windows where demand is steady but passengers are more cost-conscious, ensuring competitive rates that sustain volume during off-peak periods.

**Vendor Competitive Benchmarking**

Analysis between Creative Mobile and VeriFone reveals that Verifone INC. consistently charges a higher fare-per-mile across all hours, with the gap widening significantly during peak hours. This poses a competitive risk if passengers begin preferring Creative Mobile-dispatched vehicles for price reasons. An audit should be conducted to determine whether the premium charged by Verifone is justified by service quality differences such as vehicle type or driver ratings, or whether it represents uncontrolled surge increase. Aligning peak-hour pricing within a defined tolerance band across both vendors would ensure competitive consistency and protect market share.

**Payment Type and Tip Optimisation**

With approximately 18% of trips paid in cash, a significant portion of tip revenue remains uncaptured in the dataset and likely undertipped due to the absence of digital tracking. Introducing in-app tip suggestions and incentivising credit card payments through small fare discounts or loyalty points would gradually shift the payment mix toward card-based transactions, increasing captured tip revenue and improving overall driver earnings. 

**Group Travel and Shared Ride Pricing**

Zones with consistently high passenger counts per trip such as hotel corridors, and tourist areas present a strong case for formalised shared ride pricing tiers. Since these passengers are already travelling in groups organically, a structured multi-passenger discount that slightly reduces per-person fare while increasing total trip revenue would improve affordability perception and incentivise larger groups to choose taxis over alternatives.

**Seasonal Revenue Protection**

a targeted off-peak pricing strategy during Q1 should be started — introducing modest fare reductions on mid-day and weekend leisure trips to increase the demand, while maintaining full peak pricing during commuter windows that remain in high demand. This will reduce the impact of seasonal downturns on overall annual performance.